<a href="https://colab.research.google.com/github/gcc712-lgtm/bigdata/blob/main/week06_5756300_%EA%B9%80%EC%B1%84%EC%9B%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6주차 과제
빅데이터프로그래밍 · 통계학과

**이름:김채원**
**학번:5756300**

## 과제 내용
은닉층이 1개인 모델과 2개인 모델을 각각 작성하고 구조 차이를 설명합니다.

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week06_학번_이름.ipynb` 로 제출합니다.


In [ ]:
import torch
import torch.nn as nn
import pandas as pd

torch.manual_seed(0)


## 문제 1. 은닉층 1개 모델 (25점)
`nn.Module` 을 상속해 아래 조건의 모델을 작성하세요.

- 입력 특성 5개, 은닉층 뉴런 16개, 출력 3개
- `__init__` 과 `forward` 를 모두 구현하고 각 줄에 주석
- `print(model)` 로 구조 출력


In [ ]:
# 답안
class SimpleNet(nn.Module): # nn.Module을 상속받아 신경만 클래스 정의
    # 모델 계층 구조를 정의하는 초기화 메서드
    def __init__(self, in_features=5, hidden=16, out_features=3):
        super().__init__() # 부모 클래스의 초기화 메서드 호출

        # 입력 특성을 은닉층 뉴런으로 연결하는 선형 계층
        self.fc1 = nn.Linear(in_features, hidden)

        # 비선형성을 부여하기 위한 ReLU 활성화 함수
        self.relu = nn.ReLU()

        # 은닉층 뉴런을 출력으로 연결하는 선형 계층
        self.fc2 = nn.Linear(hidden, out_features)

    def forward(self, x): # 입력 데이터 x가 모델을 거치는 순전파 메서드
        x = self.fc1(x) # 입력 데이터에 첫 번째 선형 계층 연산 적용
        x = self.relu(x) # 첫 번째 계층의 출력에 ReLU 활성화 함수 적용
        x = self.fc2(x) # 활성화 함수를 지난 값에 두 번째 선형 계층 연산 적용
        return x # 최종 연산 결과(출력값) 반환

model = SimpleNet() # SimpleNet 모델 객체 생성
print(model) # 생성된 모델의 계층 구조 출력

SimpleNet(
  (fc1): Linear(in_features=5, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=3, bias=True)
)


## 문제 2. 은닉층 2개 모델 (25점)
문제 1과 같은 입력·출력에 은닉층을 2개로 늘린 모델을 작성하고 구조를 출력하세요.


In [ ]:
# 답안
class TwoHidden(nn.Module):
    def __init__(self):
        super(TwoHidden, self).__init__()
        self.fc1 = nn.Linear(5, 16)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(16, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

model = TwoHidden()
print(model)

TwoHidden(
  (fc1): Linear(in_features=5, out_features=16, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=16, out_features=16, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=16, out_features=3, bias=True)
)


## 문제 3. 두 모델의 구조 비교표 (30점)
아래 항목으로 표를 만들어 출력하세요.

| 항목 | 은닉층 1개 | 은닉층 2개 |
| --- | --- | --- |
| Linear 층 수 | | |
| 전체 파라미터 수 | | |
| 활성화 함수 개수 | | |
| 출력 shape | | |


In [ ]:
# 답안 — pd.DataFrame 으로 만들어 출력하세요
rows = []
for name, m in [("은닉층 1개", SimpleNet()), ("은닉층 2개", TwoHidden())]:
    rows.append({
        "항목": name,
        "Linear 층 수": sum(1 for I in m.modules() if isinstance(I, nn.Linear)),
        "전체 파라미터 수": sum(p.numel() for p in m.parameters()),
        "활성화 함수 개수": sum(1 for I in m.modules() if isinstance(I, nn.ReLU)),
        "출력 shape": tuple(m(torch.randn(1,5)).shape),
    })

pd.DataFrame(rows).set_index("항목").T

항목,은닉층 1개,은닉층 2개
Linear 층 수,2,3
전체 파라미터 수,147,419
활성화 함수 개수,1,2
출력 shape,"(1, 3)","(1, 3)"


## 문제 4. 구조 차이 설명 (20점)
아래 세 가지를 각각 두세 줄로 설명하세요.

1. 층을 늘리면 파라미터 수가 왜 늘어나는가
2. 활성화 함수가 없으면 층을 쌓아도 소용없는 이유
3. 층을 늘리는 것이 항상 좋은가 (근거를 함께)


**답:**

1. 각 층에는 입력값을 계산하기 위한 가중치와 편향이 존재한다(가중치 행렬이 새로 생김). 층이 늘어날수록 새로운 가중치와 편향이 생기므로 전체 파라미터 수가 늘어난다.

2. 활성화 함수가 없으면 각 층은 선형 변환일 뿐이다. 선형 변환을 여러 번 합성해도 결국 하나의 선형 변환과 같기 때문이다.

3. 아니다. 층이 지나치게 깊어지면 훈련 데이터에만 과도하게 적합되어 과적합이 일어날 수 있다. 또한, 기울기 소실과 파라미터 수로 인해 메모리와 연산 속도가 늘어나는 단점이 발생한다.


## 문제 5. 임의 입력으로 확인 (보너스)
두 모델에 각각 `torch.randn(8, 5)` 를 넣어 출력 shape을 확인하고, 중간 층의 shape 변화를 출력하세요.


In [ ]:
# 답안
model1 = SimpleNet()
model2 = TwoHidden()

x = torch.randn(8, 5)

print("SimpleNet Shape 변화")
x1 = model1.fc1(x)
print("fc1 출력:", x1.shape)
x1 = model1.relu(x1)
print("ReLU 출력:", x1.shape)
x1 = model1.fc2(x1)
print("fc2 출력:", x1.shape)
print("최종 출력:", x1.shape)


print("TwoHidden Shape 변화")
x2 = model2.fc1(x)
print("fc1 출력:", x2.shape)
x2 = model2.relu1(x2)
print("ReLU1 출력:", x2.shape)
x2 = model2.fc2(x2)
print("fc2 출력:", x2.shape)
x2 = model2.relu2(x2)
print("ReLU2 출력:", x2.shape)
x2 = model2.fc3(x2)
print("fc3 출력:", x2.shape)
print("최종 출력:", x2.shape)

SimpleNet Shape 변화
fc1 출력: torch.Size([8, 16])
ReLU 출력: torch.Size([8, 16])
fc2 출력: torch.Size([8, 3])
최종 출력: torch.Size([8, 3])
TwoHidden Shape 변화
fc1 출력: torch.Size([8, 16])
ReLU1 출력: torch.Size([8, 16])
fc2 출력: torch.Size([8, 16])
ReLU2 출력: torch.Size([8, 16])
fc3 출력: torch.Size([8, 3])
최종 출력: torch.Size([8, 3])
